# YOLO26 2차 실험 (Colab)

외부 데이터(AI Hub)가 들어간 2차 데이터로 학습. 1차 원본은 `exp1/yolo26-exp1.ipynb`.
모델이랑 설정은 1차 그대로 두고 데이터만 바꿈. 진행 방법은 레포 `EXPERIMENTS.md` Colab 절 기준.

| 항목 | 값 |
|---|---|
| 방법 × 에폭 | v2, v3, v4 × 5, 10, 20 (각각 처음부터) |
| 모델 | yolo26s, COCO 사전학습에서 전체 학습 |
| imgsz / batch / seed | 640 / 16 / 0 |
| optimizer / lr0 | AdamW / 0.000167 (1차에서 auto가 실제로 고른 값) |
| 나머지 | ultralytics 기본값 (1차랑 같음) |
| 비교 점수 | val, kaggle, test_only (셋 다 대회 지표 mAP@[0.75:0.95]) |

## 1. 세션 시작

드라이브 연결, 팀 레포 받기, 패키지 설치, W&B 로그인. 세션마다 다시 실행.
W&B 팀은 로그인보다 먼저 정해야 팀 프로젝트에 기록됨.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

ZIP_DIR = '/content/drive/MyDrive/pill data/data'   # zip 4개 있는 곳
SAVE_DIR = '/content/drive/MyDrive/pill data/runs'  # 내 결과 폴더 (1차랑 같은 곳)
REPO = '/content/pill-object-detection'
BRANCH = 'main'  # 공용 코드는 main 걸 씀

# 팀 레포 받기 (이미 있으면 최신으로만)
!test -d "{REPO}" || git clone -q https://github.com/codeit-ai-15th/pill-object-detection.git "{REPO}"
%cd {REPO}
!git fetch -q origin {BRANCH} && git switch -q -C {BRANCH} FETCH_HEAD && git log -1 --format='%h %s'

# 팀 uv.lock이랑 같은 버전 + 제출용 kaggle
!pip install -q ultralytics==8.4.183 pycocotools==2.0.11 wandb==0.31.0 kaggle

import sys
sys.path.insert(0, f'{REPO}/src')  # 팀 공용 코드(pill_detection) 불러오기용

import os, wandb
os.environ['WANDB_ENTITY'] = 'codeit-prj1-team2'    # 로그인보다 먼저
wandb.login(key=userdata.get('WANDB_API_KEY'))      # 키는 1차처럼 보안 비밀에서

import torch, ultralytics
print('ultralytics', ultralytics.__version__, '| GPU', torch.cuda.get_device_name(0))

## 1-2. Kaggle 연결

1차랑 같은 방식. 키는 보안 비밀 KAGGLE_JSON. 세션마다 다시 실행.
제출 csv는 팀 가이드대로 runs/submissions/에 모아둠.

In [ ]:
import json, subprocess
from pathlib import Path

COMP = 'ai15-level1-project'
SUB_DIR = Path(SAVE_DIR, 'submissions')  # 제출 csv 모아두는 곳

def kaggle_login():
    # 보안 비밀의 kaggle.json을 파일로 (세션마다 새로 필요)
    cred = json.loads(userdata.get('KAGGLE_JSON'))
    d = Path.home() / '.kaggle'
    d.mkdir(exist_ok=True)
    (d / 'kaggle.json').write_text(json.dumps(cred))
    os.chmod(d / 'kaggle.json', 0o600)

def run_kaggle(args):
    # kaggle 명령 실행하고 결과를 셀에 찍기 (그냥 실행하면 화면에 안 나옴)
    r = subprocess.run(['kaggle', 'competitions', *args], capture_output=True, text=True)
    print(r.stdout or r.stderr)
    r.check_returncode()  # 실패하면 여기서 오류

def submit(run_name, msg=''):
    path = SUB_DIR / f'exp2_{run_name}.csv'
    run_kaggle(['submit', '-c', COMP, '-f', str(path), '-m', f'{run_name} {msg}'.strip()])

def check_scores():
    run_kaggle(['submissions', '-c', COMP])

kaggle_login()
check_scores()  # 1차 제출 목록이 보이면 연결된 것

## 2. 데이터 풀기

드라이브 zip을 레포 안에 풂. 세션마다 다시 해야 함 (/content는 세션 끝나면 지워짐).
aihub_images.zip이 21.6GB라 오래 걸림. test_images는 제출할 때 따로 풂.

In [ ]:
for z in ['kaggle', 'aihub_annotations', 'aihub_images']:
    !unzip -q -o "{ZIP_DIR}/{z}.zip" -d "{REPO}"

!ls {REPO}/data/processed/ai_hub/train_images | wc -l   # 11974면 다 풀린 것
!df -h /content                                        # 남은 디스크 확인

### 2-1. 데이터가 팀이랑 같은지 확인 (첫 세션만)

마지막 두 줄이 "같음"이어야 함. "다름"이면 zip 다시 풀기.

In [ ]:
!PYTHONPATH=src python scripts/make_dataset_lists.py --check

## 3. 방법별 학습 폴더 만들기 (세션마다)

v2, v3, v4 학습 폴더를 /content/exp2/ 아래에 만듦. 사진은 복사 안 하고 원본에 하드링크.
경로는 세션마다 똑같이 써야 끊긴 학습을 이어서 할 수 있음.

In [ ]:
for v in ['v2', 'v3', 'v4']:
    !PYTHONPATH=src python scripts/build_dataset.py --variant {v} --out /content/exp2/{v}

!df -h /content   # 디스크가 거의 그대로여야 함

## 4. 학습 준비

1차 학습 설정 그대로. 1차는 optimizer=auto였는데, auto가 실제로 고른 값 4개를 직접 적음.
(auto로 두면 클래스 수마다 lr이 달라짐: 118, 96, 75)
run 이름은 팀 규칙대로 <방법>_<모델>_e<에폭>_s<시드>. 폴더 이름이 그대로 W&B 이름이 됨.
W&B가 팀 프로젝트에 안 붙으면 첫 에폭 전에 학습을 멈춤.

In [ ]:
from ultralytics import YOLO, settings

settings.update({'wandb': True})

if not Path(SAVE_DIR).is_dir():  # 경로 틀려도 ultralytics는 새 폴더를 만들어버림
    raise FileNotFoundError(f'드라이브에 결과 폴더가 없음: {SAVE_DIR}')

MODEL = 'yolo26s'
SEED = 0
# 1차에서 auto가 실제로 쓴 값 (ultralytics 8.4.183 소스 확인)
OPT = dict(optimizer='AdamW', lr0=0.000167, momentum=0.9, warmup_bias_lr=0.0)

def require_wandb(trainer):
    # 첫 에폭 직전에 불림. 팀 W&B에 안 붙었으면 멈추고, 붙었으면 태그 달기
    if wandb.run is None:
        raise RuntimeError('W&B 기록이 시작 안 됨. 런타임 다시 시작하고 1번 셀부터')
    if wandb.run.entity != 'codeit-prj1-team2':
        raise RuntimeError(f'W&B가 팀이 아닌 {wandb.run.entity}에 붙음. 런타임 다시 시작하고 1번 셀부터')
    variant = Path(trainer.args.data).parent.name  # /content/exp2/v2/data.yaml → v2
    wandb.run.tags = sorted({*wandb.run.tags, 'exp2', variant})

def train_one(variant, epochs):
    run = Path(SAVE_DIR, f'{variant}_{MODEL}_e{epochs}_s{SEED}')
    assert not run.exists(), f'{run} 이미 있음. 이어서 할 거면 resume, 새로 할 거면 이름 바꾸기'

    model = YOLO(f'{MODEL}.pt')
    model.add_callback('on_train_start', require_wandb)
    model.train(
        data=f'/content/exp2/{variant}/data.yaml',
        project='pill-object-detection',  # W&B 프로젝트 이름으로 쓰임
        save_dir=str(run),                # 결과는 드라이브에 바로 저장
        epochs=epochs, patience=30, imgsz=640, batch=16,
        seed=SEED, workers=8, device=0,   # e5·e10은 T4에서 2, e20은 A100이라 8 (CPU가 적으면 ultralytics가 알아서 줄임)
        cache=False,
        **OPT,
    )
    return run

print('준비 완료:', SAVE_DIR)

### 4-1. 끊긴 run 확인하고 이어서 하기

세션이 끊기면 드라이브의 last.pt에서 이어서 학습. results.csv에 끝난 에폭이 몇 개인지 먼저 봄.

In [ ]:
import pandas as pd

def check_run(run_name):
    run = Path(SAVE_DIR, run_name)
    res = run / 'results.csv'
    done = len(pd.read_csv(res)) if res.exists() else 0  # 끝난 에폭 수
    print(f'{run_name}: 끝난 에폭 {done}개 | last.pt {(run / "weights" / "last.pt").exists()}')

def resume_one(run_name):
    # last.pt에 설정이 다 들어 있어서 resume=True만 주면 같은 폴더에 이어서 저장됨
    model = YOLO(Path(SAVE_DIR, run_name, 'weights', 'last.pt'))
    model.add_callback('on_train_start', require_wandb)
    model.train(resume=True)

check_run('v2_yolo26s_e5_s0')

## 5. 학습

먼저 v2 5에폭 하나만 돌려서 실제 시간 확인. 그걸 보고 나머지 순서 정함.
5에폭이라 close_mosaic(10) 때문에 모자이크는 처음부터 꺼짐 (1차 분석 때 확인한 것).

In [ ]:
import time

t0 = time.time()
run = train_one('v2', 5)
print(f'{run.name} 끝: {(time.time() - t0) / 60:.1f}분')

## 6. val 채점

팀 채점 함수(pill_detection.evaluate)로 점수 3개. 셋 다 대회 지표 mAP@[0.75:0.95].
- val: val 1,000장 전체. 방법끼리 비교하는 기준
- kaggle: Kaggle 사진 47장만 (1차 val이랑 같은 사진). 1차랑 비교용
- test_only: test에만 나오는 18개 클래스. AI Hub로만 배운 약을 잘 찾는지
점수는 run 폴더 scores.json이랑 W&B Summary(exp2/val 등)에 붙음.
confusion matrix용으로 val 예측도 val_predictions.json으로 남김.

In [ ]:
import json, shutil
from pill_detection.evaluate import evaluate_val, predictions_from_ultralytics

def score_run(run_name):
    run = Path(SAVE_DIR, run_name)
    ds = Path('/content/exp2', run_name.split('_')[0])  # v2_yolo26s_e5_s0 → v2

    model = YOLO(run / 'weights' / 'best.pt')
    results = model.predict(str(ds / 'images/val'), conf=0.001, stream=True, verbose=False)
    preds = predictions_from_ultralytics(results)
    scores = evaluate_val(preds, ds / 'eval/val_gt.json')
    print(f"{run_name}: val {scores['val']:.4f}  kaggle {scores['kaggle']:.4f}  test_only {scores['test_only']:.4f}")

    (run / 'scores.json').write_text(json.dumps(scores, indent=2), encoding='utf-8')
    (run / 'val_predictions.json').write_text(json.dumps(preds), encoding='utf-8')  # confusion matrix용
    shutil.copy(ds / 'dataset_info.json', run)

    # 점수를 학습 때 W&B run에 붙이기 (팀 가이드 코드)
    run_id = max((run / 'wandb').glob('*run-*/run-*.wandb'), key=lambda f: f.stat().st_mtime).stem[4:]
    with wandb.init(entity='codeit-prj1-team2', project='pill-object-detection',
                    id=run_id, resume='allow', dir=str(run)) as wb:
        wb.summary.update({f'exp2/{k}': scores[k] for k in ('val', 'kaggle', 'test_only')})
        wb.config.update({'dataset_info': json.loads((ds / 'dataset_info.json').read_text(encoding='utf-8'))})
    return scores

### 6-1. val 누수 확인

val이 0.98로 너무 높음. 1차 때처럼 같은 약 조합이 train이랑 val에 나눠 들어갔는지 봄.
파일 이름 맨 앞(첫 _ 앞)이 약 조합.

In [ ]:
combo = lambda p: p.stem.split('_')[0]

for v in ['v2', 'v3', 'v4']:
    tr = {combo(p) for p in Path(f'/content/exp2/{v}/images/train').iterdir()}
    va = list(Path(f'/content/exp2/{v}/images/val').iterdir())
    shared = [p for p in va if combo(p) in tr]
    print(f'{v}: val {len(va)}장 중 train에도 있는 조합 {len(shared)}장')

print('파일 이름 예시:', [p.name for p in va[:3]])

#### 결과: val 누수는 아님

v2, v3, v4 모두 val 1,000장 중 train이랑 겹치는 조합 0장 → 누수 아님.
v2 e5가 val 0.983, kaggle 0.993인데 팀 Faster R-CNN Kaggle 실제 점수는 0.58~0.61.
1차처럼 val은 쉽고 test는 어려운 상황으로 보임. 이유는 Kaggle 제출 점수 나오면 다시 봄.

## 7. 나머지 학습 (자동으로 이어감)

순서: e5 v3 → v4, 그다음 e10 v2 → v3 → v4. 짧은 것부터 방법 전체를 채움.
끊기면 1~4-1번 셀 다시 실행하고 이 셀만 다시 돌리면 됨. 끝난 건 건너뛰고 끊긴 건 이어서 함.

In [ ]:
import time

QUEUE = [('v3', 5), ('v4', 5), ('v2', 10), ('v3', 10), ('v4', 10)]

def run_queue(queue):
    for v, ep in queue:
        name = f'{v}_{MODEL}_e{ep}_s{SEED}'
        run = Path(SAVE_DIR, name)

        if (run / 'scores.json').exists():
            print(f'[건너뜀] {name}: 채점까지 끝남')
            continue

        res = run / 'results.csv'
        done = len(pd.read_csv(res)) if res.exists() else 0  # 끝난 에폭 수
        t0 = time.time()

        if not run.exists():
            train_one(v, ep)          # 처음부터
        elif done < ep:
            print(f'[이어서] {name}: {done}/{ep} 에폭부터')
            resume_one(name)          # 끊긴 데서부터
        score_run(name)               # 학습 끝났으면 채점만
        print(f'{name} 끝: {(time.time() - t0) / 60:.1f}분\n')

run_queue(QUEUE)

#### e20 첫 시도 정리

7번 셀을 e20으로 바꿔 실행했다가 첫 에폭 전에 멈춤. 폴더만 있고 last.pt가 없어서 이어서 할 수 없음.
지우지 않고 _aborted로 이름만 바꿈. 다시 실행해도 안전함 (이미 바꿨거나 last.pt가 있으면 아무것도 안 함).

In [ ]:
old = Path(SAVE_DIR, 'v2_yolo26s_e20_s0')
dst = old.with_name(old.name + '_aborted')

if not old.exists():
    print('폴더 없음: 이미 처리됨')
elif (old / 'weights' / 'last.pt').exists():
    print('last.pt 있음: 이어서 할 수 있으니 이름 안 바꿈')
elif dst.exists():
    print(f'{dst.name}이 이미 있음: 확인 필요')
else:
    print('안에 있는 파일:', sorted(p.name for p in old.rglob('*') if p.is_file()))
    old.rename(dst)  # 지우지 않고 이름만 바꿈
    print('이름 바꿈 →', dst.name)

### 7-1. e20 (A100)

e5, e10이랑 같은 설정. 다른 점은 GPU(T4 → A100)랑 workers(2 → 8). 둘 다 학습 설정은 아니지만 결과가 아주 조금 달라질 수 있음.
e20은 앞 10에폭에 모자이크가 켜짐.

In [ ]:
import time

def run_queue(queue):
    for v, ep in queue:
        name = f'{v}_{MODEL}_e{ep}_s{SEED}'
        run = Path(SAVE_DIR, name)

        if (run / 'scores.json').exists():
            print(f'[건너뜀] {name}: 채점까지 끝남')
            continue

        res = run / 'results.csv'
        done = len(pd.read_csv(res)) if res.exists() else 0  # 끝난 에폭 수
        t0 = time.time()

        if not run.exists():
            train_one(v, ep)          # 처음부터
        elif done < ep:
            print(f'[이어서] {name}: {done}/{ep} 에폭부터')
            resume_one(name)          # 끊긴 데서부터
        score_run(name)               # 학습 끝났으면 채점만
        print(f'{name} 끝: {(time.time() - t0) / 60:.1f}분\n')

run_queue([('v2', 20), ('v3', 20), ('v4', 20)])

## 8. 결과 모아보기

드라이브 run 폴더에서 끝난 에폭 수랑 채점 점수만 읽음. GPU 없이 돌아감.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import pandas as pd
from pathlib import Path

SAVE_DIR = Path('/content/drive/MyDrive/pill data/runs')

rows = []
for run in sorted(SAVE_DIR.glob('v*_yolo26s_e*_s0')):
    res, sc = run / 'results.csv', run / 'scores.json'
    s = json.loads(sc.read_text()) if sc.exists() else {}
    rows.append({
        'run': run.name,
        '끝난 에폭': len(pd.read_csv(res)) if res.exists() else 0,
        'val': s.get('val'), 'kaggle': s.get('kaggle'), 'test_only': s.get('test_only'),
    })
print(pd.DataFrame(rows).to_string(index=False))

## 9. Kaggle 제출 (GPU 없이 가능)

GPU 한도 걸려서 CPU 런타임으로 함. 학습 데이터는 안 풀고 test 사진만 풂.
대회 클래스 목록은 레포 data/datasets/classes.csv (학습 폴더 classes.csv랑 같은 파일).
val이 거의 만점이라 방법끼리 구분이 안 돼서, e10 세 개를 다 내고 실제 test 점수로 비교.

### 9-1. CPU 세션 준비

GPU 한도 걸려서 CPU 런타임으로 연결. 학습 데이터는 안 풀고 레포, 패키지, test 사진만 준비.
이 셀 다음에 1-2번(Kaggle 연결)을 다시 실행해야 submit, check_scores를 쓸 수 있음.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
ZIP_DIR = '/content/drive/MyDrive/pill data/data'
SAVE_DIR = '/content/drive/MyDrive/pill data/runs'
REPO = '/content/pill-object-detection'

!test -d "{REPO}" || git clone -q https://github.com/codeit-ai-15th/pill-object-detection.git "{REPO}"
%cd {REPO}
!git fetch -q origin main && git switch -q -C main FETCH_HEAD && git log -1 --format='%h %s'
!pip install -q ultralytics==8.4.183 pycocotools==2.0.11 kaggle
sys.path.insert(0, f'{REPO}/src')

# test 사진만 풀기
!unzip -q -o "{ZIP_DIR}/test_images.zip" -d "{REPO}"
!ls {REPO}/data/raw/test_images | wc -l   # 842면 OK

### 9-2. 제출 파일 만들기 (e10 세 개)

make_submission: best.pt로 test 842장 예측해서 runs/submissions/exp2_<run>.csv로 저장.
대회 클래스 목록은 레포 data/datasets/classes.csv. conf 0.01, 대회 밖 클래스 빼는 건 팀 가이드 그대로.
CPU라 run 하나에 몇 분씩 걸림.

In [ ]:
from ultralytics import YOLO
from pill_detection.evaluate import predictions_from_ultralytics, write_submission, competition_from_classes

COMPETITION = competition_from_classes(f'{REPO}/data/datasets/classes.csv')
print('대회 클래스:', len(COMPETITION))  # 74개여야 함
TEST_DIR = f'{REPO}/data/raw/test_images'

def make_submission(run_name, device='cpu'):  # GPU 있으면 device=0
    model = YOLO(Path(SAVE_DIR, run_name, 'weights', 'best.pt'))
    results = model.predict(TEST_DIR, conf=0.01, device=device, stream=True, verbose=False)
    out = Path(SAVE_DIR, 'submissions', f'exp2_{run_name}.csv')
    n = write_submission(predictions_from_ultralytics(results), out, COMPETITION)
    print(f'{run_name}: {n}줄 → {out}')

# 지금 만들 것: e10 세 개
for name in ['v2_yolo26s_e10_s0', 'v3_yolo26s_e10_s0', 'v4_yolo26s_e10_s0']:
    make_submission(name)

### 9-3. e10 세 개 제출

val이 방법끼리 구분을 못 해서, e10 세 개를 다 내고 실제 test 점수로 비교.
팀이 하루 10번을 나눠 써서 내 몫은 5번. 여기서 3번 씀.

In [ ]:
# 내 제출 몫 5번. 이번에 3번 사용 → 남은 2번은 v3 e5, v2 e5에 씀 (아래 두 셀)
TO_SUBMIT = ['v2_yolo26s_e10_s0', 'v3_yolo26s_e10_s0', 'v4_yolo26s_e10_s0']
for name in TO_SUBMIT:
    submit(name, '2차 YOLO26s')
check_scores()

### 9-4. v3 e5 제출

e10 중 이름상 1등인 v3로 e5도 냄. 에폭 5랑 10이 test에서 차이 나는지 보려고. 4번째 제출.

In [ ]:
make_submission('v3_yolo26s_e5_s0')
submit('v3_yolo26s_e5_s0', '2차 YOLO26s')
check_scores()

### 9-5. v2 e5 제출

v3에서 e5가 e10보다 높게 나와서, 같은 조건(둘 다 모자이크 꺼짐)으로 다른 방법에서도 그런지 확인. 마지막 5번째 제출.
e20 대신 이걸 고른 이유는 e20은 모자이크가 앞 10에폭에 켜져서 길이랑 섞이는 비교라서.

In [ ]:
make_submission('v2_yolo26s_e5_s0')
submit('v2_yolo26s_e5_s0', '2차 YOLO26s')
check_scores()

#### 결과: Kaggle 제출

| run | val | Kaggle |
|---|---|---|
| v2 e5 | 0.9830 | 0.5912 |
| v2 e10 | 0.9878 | 0.6019 |
| v3 e5 | 0.9846 | 0.6133 |
| v3 e10 | 0.9901 | 0.6035 |
| v4 e10 | 0.9889 | 0.6007 |

- 1차 최고 0.410 → 2차 0.59~0.61. 외부 데이터 효과 확인
- e10끼리 방법 차이 0.003. 1차 시드 흔들림(0.02)보다 작아서 우열 없음
- e5 vs e10: v3는 e5가 +0.0098, v2는 e5가 −0.0107. 방향이 엇갈림 → 에폭 차이도 흔들림 수준
- e5, e10은 둘 다 모자이크가 처음부터 꺼진 같은 조건이라, 이 비교엔 모자이크 영향 없음
- val은 0.98~0.99로 거의 만점인데 test는 0.60. val로는 방법도 에폭도 못 고름
- 팀장님 Faster R-CNN도 0.59~0.61 → 방법, 에폭, 모델보다 데이터 쪽 한계로 보임
- e20은 이날 제출 몫이 없어서 안 냄. 이후 A100으로 학습(7-1)하고 제출 파일 만듦(12번), 제출은 다음 날

## 10. 결과 정리용 숫자 모으기

run별 precision, recall, mAP(ultralytics, val, IoU 0.5), 대회 지표 점수, 클래스별 AP,
제출 파일의 사진당 박스 수(score 0.5 이상)를 한 파일로 모음. GPU 없이 됨.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import pandas as pd
from pathlib import Path

SAVE_DIR = Path('/content/drive/MyDrive/pill data/runs')
KAGGLE = {  # 제출 기록에서 옮김
    'v2_yolo26s_e5_s0': 0.59123, 'v2_yolo26s_e10_s0': 0.60190,
    'v3_yolo26s_e5_s0': 0.61333, 'v3_yolo26s_e10_s0': 0.60353,
    'v4_yolo26s_e10_s0': 0.60067,
}
P, R, M50, M = 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)'

out = {'runs': {}, 'per_class': {}, 'test_boxes': {}}
for run in sorted(SAVE_DIR.glob('v*_yolo26s_e*_s0')):
    res = pd.read_csv(run / 'results.csv')
    res.columns = res.columns.str.strip()
    best = res.loc[res[M].idxmax()]  # best.pt가 저장된 에폭
    sc = json.loads((run / 'scores.json').read_text())

    out['runs'][run.name] = {
        'epochs': len(res), 'best_epoch': int(best['epoch']),
        'precision': float(best[P]), 'recall': float(best[R]),
        'mAP50': float(best[M50]), 'mAP50_95': float(best[M]),
        'val': sc['val'], 'kaggle47': sc['kaggle'], 'test_only': sc['test_only'],
        'kaggle_public': KAGGLE.get(run.name),
    }
    out['per_class'][run.name] = sc['per_class']  # 대회 클래스별 AP

    sub = SAVE_DIR / 'submissions' / f'exp2_{run.name}.csv'
    if sub.exists():  # 제출한 run만
        s = pd.read_csv(sub)
        conf = s[s['score'] >= 0.5]
        per_img = conf.groupby('image_id').size()
        dist = {int(k): int(v) for k, v in per_img.value_counts().sort_index().items()}
        dist[0] = 842 - len(per_img)  # 자신 있는 박스가 하나도 없는 사진
        out['test_boxes'][run.name] = {
            'n_boxes': int(len(conf)),
            'per_image': dict(sorted(dist.items())),
            'by_class': {int(k): int(v) for k, v in conf['category_id'].value_counts().items()},
        }

path = SAVE_DIR / 'exp2_summary.json'
path.write_text(json.dumps(out, ensure_ascii=False, indent=1), encoding='utf-8')
print(pd.DataFrame(out['runs']).T[['precision', 'recall', 'mAP50', 'mAP50_95', 'val', 'kaggle_public']].round(4))
print('저장:', path)

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/pill data/runs/exp2_summary.json')

## 11. 1차 모델 다시 채점 (팀 가이드)

1차 기준 run(yolo26s_e100_s0_fix, Kaggle 0.410)을 2차 val로 채점. 1차랑 2차를 같은 대회 지표로 비교하려고.
1차 모델은 Kaggle 56개 클래스만 배워서 kaggle 점수만 봄 (val, test_only는 낮게 나오는 게 정상).
val은 네 방법이 같아서 v2 폴더 씀.

In [ ]:
RUN1 = Path(SAVE_DIR, 'yolo26s_e100_s0_fix')
DS = Path('/content/exp2/v2')

model = YOLO(RUN1 / 'weights' / 'best.pt')
print('클래스 이름 예:', model.names[0])  # '1900' 같은 K-코드 숫자여야 채점 가능

results = model.predict(str(DS / 'images/val'), conf=0.001, stream=True, verbose=False)
scores = evaluate_val(predictions_from_ultralytics(results), DS / 'eval/val_gt.json')
print(f"1차 기준점 kaggle: {scores['kaggle']:.4f}   (참고: val {scores['val']:.4f}, test_only {scores['test_only']:.4f})")

(RUN1 / 'scores_exp2val.json').write_text(json.dumps(scores, indent=2), encoding='utf-8')
print('저장:', RUN1 / 'scores_exp2val.json')

#### 결과: 1차 모델 다시 채점

| | Kaggle 47장 (대회 지표) | Kaggle public |
|---|---|---|
| 1차 기준 (yolo26s_e100_s0_fix) | 0.9615 | 0.410 |
| 2차 (9개 run) | 0.9906–0.9983 | 0.59–0.61 |

- 클래스 이름이 K-코드 숫자(1900)라 2차 채점 함수로 바로 잴 수 있었음
- 같은 47장에서 1차 → 2차는 +0.03~0.04. 근데 Kaggle public은 +0.19
- 47장은 1차 val 사진이라 train에 있는 30종만 나옴. 1차 모델도 이미 0.96이라 외부 데이터 효과(test 전용 18종)가 거의 안 잡힘
- val 0.6460, test_only 0.0000은 1차 모델이 test 전용 약을 안 배워서 나오는 정상 값. 가이드대로 kaggle 점수만 봄
- 점수는 1차 run 폴더 scores_exp2val.json에 저장

## 12. e20 제출 파일 만들기

e20 세 개 test 예측해서 캐글 제출 csv 만듦. 제출은 다음 날 (오늘 제출 몫 다 씀).
9-1, 9-2를 다시 돌리면 기존 출력이 바뀌고 e10 파일도 다시 만들어서, 필요한 것만 여기 모음.
conf 0.01, 대회 밖 클래스 빼는 건 팀 가이드 그대로.

In [ ]:
# test 사진 풀기
!unzip -q -o "{ZIP_DIR}/test_images.zip" -d "{REPO}"
!ls {REPO}/data/raw/test_images | wc -l   # 842면 OK

from pill_detection.evaluate import predictions_from_ultralytics, write_submission, competition_from_classes

COMPETITION = competition_from_classes(f'{REPO}/data/datasets/classes.csv')
print('대회 클래스:', len(COMPETITION))  # 74개여야 함
TEST_DIR = f'{REPO}/data/raw/test_images'

for name in ['v2_yolo26s_e20_s0', 'v3_yolo26s_e20_s0', 'v4_yolo26s_e20_s0']:
    out = Path(SAVE_DIR, 'submissions', f'exp2_{name}.csv')
    assert not out.exists(), f'{out.name} 이미 있음. 덮어쓰지 않고 멈춤'
    model = YOLO(Path(SAVE_DIR, name, 'weights', 'best.pt'))
    results = model.predict(TEST_DIR, conf=0.01, device=0, stream=True, verbose=False)
    n = write_submission(predictions_from_ultralytics(results), out, COMPETITION)
    print(f'{name}: {n}줄 → {out.name}')

In [ ]:
for f in sorted(Path(SAVE_DIR, 'submissions').glob('exp2_*.csv')):
    s = pd.read_csv(f)
    print(f'{f.name}: {len(s)}줄 | 점수 0.5 미만 {(s["score"] < 0.5).sum()}개')

## 13. e20 Kaggle 제출 (v3 e20 하나만)

11/3 회의에서 e20 1개만 내기로 함.
v3 고른 이유: e5(0.6133), e10(0.6035) 점수가 이미 있어서, e20까지 내면 한 방법의 에폭별 test 점수가 다 갖춰짐.
제출 파일은 12번에서 만든 것 그대로. 기존 셀을 다시 돌리지 않게 로그인부터 여기 다 넣음.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import json, os, subprocess
from pathlib import Path

!pip install -q kaggle

COMP = 'ai15-level1-project'
SUB = Path('/content/drive/MyDrive/pill data/runs/submissions/exp2_v3_yolo26s_e20_s0.csv')
assert SUB.exists(), f'제출 파일 없음: {SUB}'

# Kaggle 로그인 (1-2번과 같은 방식)
cred = json.loads(userdata.get('KAGGLE_JSON'))
d = Path.home() / '.kaggle'
d.mkdir(exist_ok=True)
(d / 'kaggle.json').write_text(json.dumps(cred))
os.chmod(d / 'kaggle.json', 0o600)

def run_kaggle(args):
    r = subprocess.run(['kaggle', 'competitions', *args], capture_output=True, text=True)
    print(r.stdout or r.stderr)
    r.check_returncode()

run_kaggle(['submit', '-c', COMP, '-f', str(SUB), '-m', 'v3_yolo26s_e20_s0 2차 YOLO26s'])
run_kaggle(['submissions', '-c', COMP])

#### 결과: v3 e20 Kaggle

| v3 | val | Kaggle |
|---|---|---|
| e5 | 0.9846 | 0.6133 |
| e10 | 0.9901 | 0.6035 |
| e20 | 0.9918 | 0.6339 |

- 2차 YOLO 최고점. e10보다 +0.030, e5보다 +0.021. 처음으로 흔들림(0.02)을 넘음
- e20은 에폭 말고도 모자이크(앞 10에폭 켜짐), GPU(A100), workers(8)가 달라서 에폭 효과라고 단정 못 함
- 특히 모자이크가 test에 더 잘 맞게 했을 수 있음. 3차에서 나눠서 확인해볼 가설
- v3에서 e20이 가장 높다는 건 val이랑 test가 같음. 근데 e5, e10 순서는 val(e5 < e10)이랑 test(e10 < e5)가 반대라, val을 믿을 수 있다고 보긴 어려움

## 14. confusion matrix, IoU 기준별 AP (보고서용)

### 14-1. val 정답 파일 다시 만들기

confusion matrix엔 val 정답(val_gt.json)이 필요한데, 학습 세션 임시 폴더에만 있어서 지금은 없음.
팀 코드 확인해보니 정답은 사진 없이 라벨만으로 만들어져서, 작은 zip 두 개만 풂. GPU 필요 없음.
build_dataset.py가 val_gt.json 만드는 줄을 그대로 씀 (사진 파일 있는지 확인하는 부분만 뺌).
같은 파일인지는 v3 e20을 다시 채점해서 scores.json이랑 같은지로 확인.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, json
from pathlib import Path

ZIP_DIR = '/content/drive/MyDrive/pill data/data'
SAVE_DIR = Path('/content/drive/MyDrive/pill data/runs')
REPO = '/content/pill-object-detection'

!test -d "{REPO}" || git clone -q https://github.com/codeit-ai-15th/pill-object-detection.git "{REPO}"
%cd {REPO}
!git fetch -q origin main && git switch -q -C main FETCH_HEAD && git log -1 --format='%h %s'
!pip install -q pycocotools==2.0.11
sys.path.insert(0, f'{REPO}/src')

# 라벨만 있으면 돼서 작은 zip 두 개만 (사진 zip은 안 풂)
for z in ['kaggle', 'aihub_annotations']:
    !unzip -q -o "{ZIP_DIR}/{z}.zip" -d "{REPO}"

from pill_detection import datasets as ds
from pill_detection.evaluate import evaluate_val

data_dir = Path(REPO, 'data')
lists = ds.read_image_list(data_dir / 'datasets/images.csv', 'v2')    # val은 네 방법이 같음
rows = ds.read_class_table(data_dir / 'datasets/classes.csv', 'v2')['rows']
images, _ = ds.load_images(data_dir)

# build_dataset.py랑 같은 줄 (모든 객체를 원래 K-코드로)
gt_categories = [ds._coco_category(c, rows[c])
                 for c in sorted({c for name in lists['val'] for c in ds.object_classes(images[name])})]
gt, _, _ = ds._coco(images, lists['val'], gt_categories, lambda c: c)

out = SAVE_DIR / 'exp2_val_gt.json'
out.write_text(json.dumps(gt, ensure_ascii=False), encoding='utf-8')
print('val 사진', len(gt['images']), '| 객체', len(gt['annotations']), '| 클래스', len(gt['categories']))
print('저장:', out)

# 확인: 이 정답으로 v3 e20을 다시 채점하면 scores.json이랑 같아야 함
run = SAVE_DIR / 'v3_yolo26s_e20_s0'
again = evaluate_val(json.loads((run / 'val_predictions.json').read_text()), gt)
saved = json.loads((run / 'scores.json').read_text())
print(f"다시 채점 val {again['val']:.6f} | scores.json val {saved['val']:.6f}")

#### 결과: val 정답 파일

- val 사진 1000, 객체 3790, 클래스 96. 학습 때 val 정보랑 같음
- 이 정답으로 v3 e20 다시 채점한 val 0.991759 = scores.json 0.991759 → 학습 때랑 같은 정답 파일
- runs/exp2_val_gt.json으로 저장. 다음엔 이거 바로 읽으면 됨

### 14-2. confusion matrix (v3 e20)

Kaggle 최고점 run(v3 e20)으로 1차 7-3이랑 같은 방식으로 직접 셈.
대회 밖 약으로 찍은 예측이랑 대회 밖 약에 겹친 예측은 팀 채점 함수(drop_ignored)로 먼저 뺌. 점수 계산이랑 같은 기준.
점수 0.25 이상만, 정답마다 가장 많이 겹치는 예측과 짝지음. IoU 0.5 이상 + 같은 약 = 맞춤, 다른 약 = 클래스 틀림.
그림은 헷갈린 약만 모아서 그림 (약 이름은 한글이 깨져서 K-코드 숫자로).
결과표랑 그림은 runs/figures에 저장.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pill_detection.evaluate import drop_ignored, box_iou

FIG_DIR = SAVE_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)

classes = pd.read_csv(data_dir / 'datasets/classes.csv').set_index('category_id')
name_of = lambda c: classes['name'].get(c, '?')                     # 약 이름 (표에만 씀)
competition = {c['id'] for c in gt['categories'] if c['competition']}
file_of = {im['id']: im['file_name'] for im in gt['images']}

def confusion(run_name, conf=0.25):
    preds = json.loads((SAVE_DIR / run_name / 'val_predictions.json').read_text())
    preds = [p for p in drop_ignored(preds, gt) if p['score'] >= conf]  # 팀 채점 규칙 + 자신 있는 것만

    pred_of, gt_of = {}, {}
    for p in preds:
        pred_of.setdefault(p['file_name'], []).append(p)
    for a in gt['annotations']:
        if a['category_id'] in competition:                             # 대회 클래스 정답만 셈
            gt_of.setdefault(file_of[a['image_id']], []).append(a)

    rows, fp = [], 0
    for name in file_of.values():
        P, used = pred_of.get(name, []), set()
        for a in gt_of.get(name, []):
            best, bi = 0.0, -1
            for i, p in enumerate(P):                                   # 아직 안 쓴 예측 중 제일 많이 겹치는 것
                if i not in used and box_iou(a['bbox'], p['bbox']) > best:
                    best, bi = box_iou(a['bbox'], p['bbox']), i
            if best >= 0.5:
                used.add(bi)
                pc = P[bi]['category_id']
                kind = '맞춤' if pc == a['category_id'] else '클래스 틀림'
            else:
                pc, kind = None, ('박스 어긋남' if best > 0.1 else '못 찾음')
            rows.append({'사진': name, '정답': a['category_id'], '예측': pc, 'IoU': round(best, 3), '결과': kind})
        fp += len(P) - len(used)                                         # 짝 못 찾은 예측
    return pd.DataFrame(rows), fp

RUN = 'v3_yolo26s_e20_s0'
res, fp = confusion(RUN)
print(res['결과'].value_counts().to_string())
print('없는 걸 찾음:', fp, '| 정답 박스:', len(res))

# 헷갈린 약 쌍
wrong = res[res['결과'] == '클래스 틀림']
pairs = wrong.groupby(['정답', '예측']).size().rename('개수').reset_index().sort_values('개수', ascending=False)
pairs['정답 이름'] = pairs['정답'].map(name_of)
pairs['예측 이름'] = pairs['예측'].map(name_of)
print('\n[헷갈린 약 쌍]')
print(pairs.to_string(index=False) if len(pairs) else '없음')
res.to_csv(FIG_DIR / f'exp2_confusion_{RUN}.csv', index=False, encoding='utf-8-sig')

# 그림: 헷갈린 약만 모은 작은 confusion matrix
if len(pairs):
    ids = sorted(set(pairs['정답']) | set(pairs['예측']))
    sub = res[res['정답'].isin(ids) & res['예측'].isin(ids)]
    mat = pd.crosstab(sub['정답'], sub['예측']).reindex(index=ids, columns=ids, fill_value=0)
    fig, ax = plt.subplots(figsize=(1 + 0.6 * len(ids), 1 + 0.5 * len(ids)))
    ax.imshow(mat.values, cmap='Blues')
    ax.set_xticks(range(len(ids)), [str(c) for c in ids], rotation=90)
    ax.set_yticks(range(len(ids)), [str(c) for c in ids])
    for i in range(len(ids)):
        for j in range(len(ids)):
            if mat.values[i, j]:
                ax.text(j, i, mat.values[i, j], ha='center', va='center',
                        color='white' if mat.values[i, j] > mat.values.max() / 2 else 'black')
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_title(f'{RUN}: confused classes only')
    plt.tight_layout()
    plt.savefig(FIG_DIR / f'exp2_confusion_{RUN}.png', dpi=150)
    plt.show()
print('저장:', FIG_DIR)

#### 결과: confusion matrix (v3 e20)

| 결과 | 개수 |
|---|---|
| 맞춤 | 3,052 / 3,053 |
| 클래스 틀림 | 0 |
| 박스 어긋남 | 0 |
| 못 찾음 | 1 |
| 없는 걸 찾음 | 0 |

- 대회 클래스 정답 3,053개 = val 정답 3,790개 − 대회 밖 737개 (데이터 확인 때 737개랑 같음)
- 헷갈린 약 쌍 0개라 그림은 안 만들어짐
- 못 찾은 1개는 울트라셋이알서방정(27926)
- 1차(val 47장)는 159개 중 맞춤 149, 클래스 틀림 6. val 구성이 달라 직접 비교는 조심

### 14-3. IoU 기준별 AP (v3 e20)

보고서 초안의 "AP95에서 특히 낮아지는 클래스와 그 빈도"를 쓰려고 계산.
팀 coco_map이랑 같은 설정으로 계산하되, 다섯 기준(IoU 0.75~0.95)을 평균 안 내고 따로 꺼냄.
기준별 값을 평균 내면 scores.json의 클래스별 AP랑 같아야 함 (검증 줄).

In [ ]:
import contextlib, io
import numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pill_detection.evaluate import IOU_THRESHOLDS

print('[14-2에서 못 찾은 1개]')
print(res[res['결과'] != '맞춤'].to_string(index=False))

def ap_by_iou(run_name):
    preds = drop_ignored(json.loads((SAVE_DIR / run_name / 'val_predictions.json').read_text()), gt)
    image_of = {im['file_name']: im['id'] for im in gt['images']}
    results = [{'image_id': image_of[p['file_name']], 'category_id': p['category_id'],
                'bbox': list(p['bbox']), 'score': p['score']} for p in preds]
    cat_ids = sorted({a['category_id'] for a in gt['annotations']} & competition)
    extra = sorted({r['category_id'] for r in results} - {c['id'] for c in gt['categories']})
    g = {**gt, 'categories': [{'id': c['id']} for c in gt['categories']] + [{'id': c} for c in extra]}
    with contextlib.redirect_stdout(io.StringIO()):                   # 팀 coco_map이랑 같은 설정
        cg = COCO(); cg.dataset = g; cg.createIndex()
        E = COCOeval(cg, cg.loadRes(results), 'bbox')
        E.params.iouThrs = IOU_THRESHOLDS
        E.params.catIds = cat_ids
        E.evaluate(); E.accumulate()
    prec = E.eval['precision'][:, :, :, 0, -1]                        # [IoU 기준, recall, 클래스]
    out = {}
    for k, c in enumerate(E.params.catIds):
        out[int(c)] = [float(v[v > -1].mean()) if (v > -1).any() else 0.0
                       for v in (prec[t, :, k] for t in range(len(IOU_THRESHOLDS)))]
    return pd.DataFrame(out, index=[f'AP{round(t * 100)}' for t in IOU_THRESHOLDS]).T

ap = ap_by_iou(RUN)
AP_COLS = list(ap.columns)                                            # AP75 ~ AP95
by_iou = ap[AP_COLS].mean()                                           # 기준별 74개 클래스 평균

# 검증: 기준별 평균 = scores.json 클래스별 AP
saved = {int(k): v for k, v in json.loads((SAVE_DIR / RUN / 'scores.json').read_text())['per_class'].items()}
print(f"\n검증: scores.json 클래스별 AP와 최대 차이 {max(abs(ap.loc[c, AP_COLS].mean() - v) for c, v in saved.items()):.1e}")

print('\n[IoU 기준별 평균 AP]')
print(by_iou.round(4).to_string())

ap['75→95 감소'] = ap['AP75'] - ap['AP95']
ap['train 개수'] = classes['n_train_v3'].reindex(ap.index)
ap['이름'] = ap.index.map(name_of)
print('\n[AP95에서 가장 많이 떨어지는 클래스 10개]')
print(ap.sort_values('75→95 감소', ascending=False).head(10).round(3).to_string())
print('\n감소폭과 train 개수 상관:', round(np.corrcoef(ap['75→95 감소'], ap['train 개수'])[0, 1], 3))
ap.round(4).to_csv(FIG_DIR / f'exp2_ap_by_iou_{RUN}.csv', encoding='utf-8-sig')

# 그림: IoU 기준별 평균 AP
plt.figure(figsize=(5, 3.5))
plt.plot([75, 80, 85, 90, 95], by_iou.values, marker='o')
plt.xlabel('IoU threshold (%)'); plt.ylabel('mean AP (competition classes)')
plt.title(f'{RUN}: AP by IoU threshold'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(FIG_DIR / f'exp2_ap_by_iou_{RUN}.png', dpi=150); plt.show()
print('저장:', FIG_DIR)

#### 결과: IoU 기준별 AP

| IoU | 0.75 | 0.80 | 0.85 | 0.90 | 0.95 |
|---|---|---|---|---|---|
| 평균 AP | 1.0000 | 1.0000 | 0.9993 | 0.9979 | 0.9617 |

- 검증: 기준별 평균이 scores.json 클래스별 AP랑 최대 1.1e-16 차이 → 팀 채점이랑 같은 계산
- 0.90까지는 거의 만점. 깎이는 건 거의 IoU 0.95 하나
- AP95에서 많이 떨어지는 클래스: 놀텍정(−0.241), 삐콤씨에프정(−0.239), 트루비타정(−0.207) 등
- 이 10개의 train 개수는 189 ~ 417개로 전체(27~1,263) 중 하위 15 ~ 82%에 고르게 퍼짐. 감소폭과 train 개수 상관 0.007 → 빈도랑 관련 없음
- 14-2에서 못 찾은 1개는 울트라셋이알서방정(27926). AP95에서도 떨어지는 쪽(−0.148)